# Semana 37 · ¿Cómo medimos la distancia a las estrellas?

Este cuaderno acompaña a la sesión de proyecto de la semana 37: la paralaje y el movimiento propio, la propagación de errores, por qué $1/p$ engaña cuando el error es grande y las distancias del catálogo HYG.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. El camino de una estrella cercana en el cielo

Simulamos el desplazamiento aparente de una estrella con paralaje $p$, a una latitud eclíptica $\beta$, con un movimiento propio $\mu$. En coordenadas eclípticas (en segundos de arco respecto a su posición media), la paralaje traza una elipse de semiejes $p$ y $p\sin\beta$, y el movimiento propio la arrastra en línea recta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def camino(p, beta_grados, mu_x, mu_y, anos=4, puntos=2000):
    t = np.linspace(0, anos, puntos)                 # años
    beta = np.radians(beta_grados)
    x = p * np.sin(2 * np.pi * t) + mu_x * t        # segundos de arco
    y = p * np.sin(beta) * np.cos(2 * np.pi * t) + mu_y * t
    return t, x, y

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, beta in zip(axs, [90, 52, 5]):
    t, x, y = camino(0.286, beta, 0, 0, anos=1)
    ax.plot(x, y); ax.set_aspect("equal"); ax.set_xlim(-0.35, 0.35); ax.set_ylim(-0.35, 0.35)
    ax.set_title(f"latitud eclíptica {beta}°"); ax.set_xlabel("″"); ax.grid(alpha=0.3)
plt.suptitle("Elipse de paralaje de una estrella a 3,5 pc (sin movimiento propio)"); plt.tight_layout(); plt.show()

In [ ]:
# 61 Cygni, de forma aproximada: paralaje 0,286″, β ≈ 52°, movimiento propio de unos 5″ por año
t, x, y = camino(0.286, 52, 4.1, 3.2, anos=4)
plt.figure(figsize=(9, 6))
plt.plot(x, y)
for ano in range(5):
    i = np.argmin(abs(t - ano))
    plt.plot(x[i], y[i], "ro"); plt.annotate(f"año {ano}", (x[i], y[i]), xytext=(5, -12), textcoords="offset points")
plt.xlabel("desplazamiento (″)"); plt.ylabel("desplazamiento (″)"); plt.gca().set_aspect("equal")
plt.title("Paralaje y movimiento propio combinados (esquema)"); plt.grid(alpha=0.3); plt.show()

El vaivén de la paralaje es pequeño comparado con el movimiento propio de 61 Cygni, que es uno de los más grandes del cielo. Por eso Bessel la eligió: un movimiento propio grande sugería que estaba cerca.

## 2. Distancias y errores

$$d = \frac{1}{p}, \qquad \frac{\sigma_d}{d} \approx \frac{\sigma_p}{p}.$$

In [ ]:
def distancia(p_mas, sigma_mas):
    d = 1000 / p_mas
    return d, d * sigma_mas / p_mas

for nombre, p, s in [("Sirio (Hipparcos)", 379.2, 1.6), ("61 Cygni", 286.0, 0.1),
                     ("una estrella con p = 5 mas ± 1 mas", 5.0, 1.0), ("la misma con Gaia, ± 0,02 mas", 5.0, 0.02)]:
    d, sd = distancia(p, s)
    print(f"{nombre:<38} d = {d:9.3f} ± {sd:.3f} pc  ({100 * s / p:.2f} %)")

## 3. Cuando el error es grande, $1/p$ engaña

Tomamos una estrella a 1000 pc ($p = 1$ mas) y simulamos 100 000 medidas de su paralaje con distintos errores gaussianos. Para cada medida calculamos $1/p$ y miramos cómo se reparten las distancias obtenidas.

In [ ]:
rng = np.random.default_rng(37)
p_real = 1.0                                     # mas → 1000 pc
fig, axs = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
for ax, sigma in zip(axs, [0.05, 0.2, 0.5, 1.0]):
    p_medida = rng.normal(p_real, sigma, 100_000)
    negativas = (p_medida <= 0).mean()
    d = 1000 / p_medida[p_medida > 0]
    ax.hist(d[d < 6000], bins=120, color="tab:blue")
    ax.axvline(1000, color="r", lw=1)
    ax.set_title(f"σ = {sigma} mas ({100 * sigma / p_real:.0f} %)\nparalajes negativas: {100 * negativas:.0f} %")
    ax.set_xlabel("1/p (pc)")
plt.tight_layout(); plt.show()

Con un 5 % de error, las distancias se agrupan bien alrededor de 1000 pc. Con un 20 %, la distribución se tuerce: tiene una cola larga hacia distancias grandes, y su media ya no es 1000. Con un 50 % o un 100 %, salen paralajes negativas y $1/p$ no sirve. Por eso, con Gaia, los astrónomos solo usan $1/p$ directamente cuando el error relativo es pequeño (por ejemplo, menor del 10-20 %).

## 4. Las distancias del catálogo HYG

Las distancias de HYG proceden sobre todo de las paralajes de Hipparcos. Cargamos el subconjunto de la semana 35.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/alabcar/hola-universo/main/cuadernos/datos/estrellas_hyg.csv"
locales = ["cuadernos/datos/estrellas_hyg.csv", "../datos/estrellas_hyg.csv", "datos/estrellas_hyg.csv"]
ruta = next((r for r in locales if os.path.exists(r)), URL)
estrellas = pd.read_csv(ruta)
print(f"{len(estrellas)} estrellas leídas de {ruta}")

In [ ]:
d = estrellas[estrellas.dist_pc > 0].dist_pc
p_mas = 1000 / d
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
a1.hist(np.log10(d), bins=60, color="tab:green")
a1.set_xlabel("log₁₀ distancia (pc)"); a1.set_ylabel("número de estrellas"); a1.set_title("Distancias")
a2.hist(np.log10(p_mas), bins=60, color="tab:orange")
a2.set_xlabel("log₁₀ paralaje (mas)"); a2.set_title("Paralajes")
plt.tight_layout(); plt.show()
print(f"estrellas con paralaje mayor de 1″ (más cerca de 1 pc): {(p_mas > 1000).sum()}")
print(f"estrellas con paralaje mayor de 0,1″ (más cerca de 10 pc): {(p_mas > 100).sum()}")

Ninguna estrella tiene una paralaje mayor de 1″, como viste en la semana 33. El pico de la izquierda son las estrellas cercanas (las de menos de 25 pc), y la cola de la derecha, las brillantes y lejanas.

## 5. Opcional: consultar el archivo de Gaia (sin probar)

La ESA ofrece un archivo público con todos los datos de Gaia, que se consulta con un lenguaje parecido a SQL, el ADQL. Desde Python se puede usar la biblioteca `astroquery`. **Esta celda no se ha podido probar** al escribir el cuaderno, porque necesita conexión con el archivo de la ESA. Para usarla en Colab, cambia `USAR_GAIA` a `True`. Pide las 20 estrellas con mayor paralaje de la publicación de datos DR3; cuando salga la DR4 (prevista para el 2 de diciembre de 2026), bastará con cambiar el nombre de la tabla.

In [ ]:
USAR_GAIA = False

if USAR_GAIA:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astroquery"])
    from astroquery.gaia import Gaia
    consulta = """
        SELECT TOP 20 source_id, ra, dec, parallax, parallax_error, phot_g_mean_mag
        FROM gaiadr3.gaia_source
        WHERE parallax > 200
        ORDER BY parallax DESC
    """
    tabla = Gaia.launch_job(consulta).get_results().to_pandas()
    tabla["distancia_pc"] = 1000 / tabla["parallax"]
    print(tabla[["source_id", "parallax", "parallax_error", "phot_g_mean_mag", "distancia_pc"]])
else:
    print("Consulta a Gaia desactivada. Cambia USAR_GAIA a True para probarla.")

## 6. Experimenta

1. Simula el camino de la estrella de Barnard: paralaje 0,547″, latitud eclíptica unos 28°, movimiento propio de 10,4″ por año, casi todo hacia el norte. ¿Se distingue la elipse en una gráfica de 4 años?
2. En la simulación de errores, calcula la media y la mediana de $1/p$ para cada σ. ¿Cuál se aleja antes de 1000 pc?
3. ¿Cuántas estrellas del catálogo tendrían un error relativo menor del 10 % con un error de paralaje de 1 mas (Hipparcos)? ¿Y con 0,02 mas (Gaia)?

In [ ]:
# Tu experimento aquí